# C1 · Features in euros sink the KNN: where the distance goes

**Type**: Break the model · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Build a dataset in which twenty irrelevant features, expressed in large units, alone decide the distances between the points, and
measure by how much the KNN accuracy collapses and how much of the distance each group of features explains, with and without scaling.

## Dataset

Synthetic: `make_moons` (600 points, two moon-shaped classes, noise 0.25) provides two **informative** features on a scale
of about 1. We add twenty more, `junk`, of pure Gaussian noise multiplied by 1000 (like amounts in euros). `X` thus has 22
columns: the first two are informative, the other twenty are not. The target is the class of the moons.

## Educational purpose

See where the Euclidean distance goes: it is a sum of squares per feature, and the term with the largest unit
dominates it. And understand that scaling **fixes the scale, not the relevance**: a useless feature stays useless.

## Task

1. With `KNeighborsClassifier(n_neighbors=5)` and the cross-validation `cv` compute `acc_raw` (mean accuracy on the unscaled data, 22 columns).
2. Compute `acc_scaled`: the same thing inside `make_pipeline(StandardScaler(), KNeighborsClassifier(5))`.
3. Compute `acc_without_junk`: scaled pipeline on the two informative columns only (`X[:, :2]`).
4. Write `distance_share(X, n_pairs=3000, seed=0)`: draw `n_pairs` random pairs of rows (`rng.integers`) and return, for each column, the fraction of the **total squared distance** due to that column (sum over the pairs of the square of the differences of that column, divided by the sum over all the columns).
5. Compute `raw_share = distance_share(X)` and `scaled_share = distance_share(StandardScaler().fit_transform(X))`, then the total share of the two informative columns in `info_raw` and `info_scaled`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_moons
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
X_info, y = make_moons(n_samples=600, noise=0.25, random_state=0)      # two informative features, scale about 1
junk = rng.normal(size=(600, 20)) * 1000                               # twenty irrelevant features, in "euros"
X = np.column_stack([X_info, junk])
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Your work

In [ ]:
acc_raw = ...
acc_scaled = ...
acc_without_junk = ...
print(f"raw {acc_raw:.3f} | scaled {acc_scaled:.3f} | without the useless features {acc_without_junk:.3f}")

In [ ]:
def distance_share(X, n_pairs=3000, seed=0):
    ...

raw_share = ...
scaled_share = ...
info_raw = ...
info_scaled = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='For the pairs: `i, j = rng.integers(0, len(X), n_pairs), rng.integers(0, len(X), n_pairs)`; then `d2 = (X[i] - X[j]) ** 2` (n_pairs x columns matrix), `d2.sum(axis=0) / d2.sum()`. For the cross-validation: `cross_val_score(model, X, y, cv=cv).mean()`.'):
    check(acc_raw < 0.62, "without scaling the KNN is close to chance (accuracy below 0.62)", "with the feature in 'euros' the KNN should collapse")
    check(acc_scaled > acc_raw + 0.05, "scaling recovers part of the loss", "acc_scaled must exceed acc_raw")
    check(acc_without_junk > acc_scaled + 0.15,
          "but removing the useless features helps much more: scaling does not fix relevance",
          "without the junk columns the accuracy must clearly exceed the scaled one")
    check(len(raw_share) == 22 and abs(raw_share.sum() - 1) < 1e-9, "the 22 shares sum to 1", "distance_share returns one fraction per column, and the sum is 1")
    check(info_raw < 1e-5, "without scaling the two informative columns explain less than 0.001% of the distance", "info_raw = raw_share[:2].sum()")
    check(0.05 < info_scaled < 0.2, "after scaling the informative columns weigh about 2/22, that is less than 20%",
          "after scaling every column weighs about 1/22: the two informative ones about 9%")
done()

---
The commented solution is in `solutions/C_break_the_model/C1_scaling_and_knn_distances.ipynb`. Open it after you pass the check, or after you have really tried.